In [2]:
# Connect google drive
from google.colab import drive
ROOT = '/content/drive'
drive.mount(ROOT)

Mounted at /content/drive


# Import package

In [3]:
import os

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import (
    LabelEncoder, OneHotEncoder )
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer


# Data Loading

In [4]:
class ColabBaseConfig:
    """
    Base class to configure the root directory for Kaggle datasets.
    By default, the root_dir on Kaggle is always '/kaggle/input'.
    """
    def __init__(self, root_dir="/content/drive"):
        self.root_dir = root_dir

class ColabDataLoader(ColabBaseConfig):
    """
    Child class inheriting from KaggleBaseConfig.
    This class handles the core logic of loading a single file from a dataset.
    """
    def __init__(self, dataset_name, root_dir="/content/drive"):
        # Call the parent class constructor to set the root_dir
        super().__init__(root_dir)

        # Create the full path to the specific dataset directory
        self.dataset_name = dataset_name
        self.dataset_dir = os.path.join(self.root_dir, self.dataset_name)

    def load_csv(self, filename, **kwargs):
        """
        Read a single CSV file and return a pandas DataFrame.
        """
        file_path = os.path.join(self.dataset_dir, filename)

        # Check if the file exists before attempting to read
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found at path: {file_path}")

        df = pd.read_csv(file_path, **kwargs)

        print(f"Shape of {filename} is {df.shape}")
        return df


def load_multiple_data(data_loader, file_dict, **kwargs):
    """
    Standalone function to load multiple CSV files based on a dictionary mapping.
    """
    loaded_data = {}

    for var_name, filename in file_dict.items():
        loaded_data[var_name] = data_loader.load_csv(filename, **kwargs)

    print("All files loaded successfully!")
    return loaded_data

In [6]:
# 1. Initialize the DataLoader instance for your specific dataset
data_loader = ColabDataLoader(dataset_name="MyDrive/home-credit-default-risk")

# 2. Define your dictionary mapping variable names to their corresponding filenames
files_to_load = {
    "app_train": "application_train.csv",
    "app_test": "application_test.csv",
    "app_pre": "previous_application.csv",
    "bureau": "bureau.csv"

}

# 3. Call the standalone function, passing the loader instance and the dictionary
# This returns a dictionary with your DataFrames attached to the variable names
data = load_multiple_data(data_loader=data_loader,
                             file_dict=files_to_load)

# 4. Access your DataFrames using the keys
app_train = data["app_train"]
app_test = data["app_test"]
app_pre = data["app_pre"]

Shape of application_train.csv is (307511, 122)
Shape of application_test.csv is (48744, 121)
Shape of previous_application.csv is (1670214, 37)
Shape of bureau.csv is (1716428, 17)
All files loaded successfully!


# Preprocessing

## Utils Class/Function

In [ ]:

class BaseDataProcessor:
    """
    Parent class to handle basic DataFrame inspection such as shape and memory usage.
    """
    def __init__(self, df: pd.DataFrame):
        # Store a copy of the dataframe to avoid setting with copy warning
        self.df = df.copy()

    def get_shape(self):
        """Returns the shape of the DataFrame."""
        return self.df.shape

    def get_memory_usage(self):
        """Returns the memory usage of the DataFrame in Megabytes (MB)."""
        # deep=True ensures accurate memory calculation for object types
        return self.df.memory_usage(deep=True).sum() / (1024 ** 2)


class ReduceMemProcessor(BaseDataProcessor):
    """
    Child class dedicated to reducing memory usage of a pandas DataFrame
    by downcasting numeric types and converting low-cardinality objects to categories.
    """
    def __init__(self, df: pd.DataFrame):
        super().__init__(df)

    def process(self, verbose=True):
        """
        Iterates through all columns and downcasts data types to the smallest possible safe type.
        """
        start_mem = self.get_memory_usage()

        for col in self.df.columns:
            col_type = self.df[col].dtype

            # Process numeric columns (Exclude object and existing categorical types)
            if col_type != object and not isinstance(col_type, pd.CategoricalDtype):
                c_min = self.df[col].min()
                c_max = self.df[col].max()

                # Downcast Integer types
                if str(col_type)[:3] == 'int':
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        self.df[col] = self.df[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        self.df[col] = self.df[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        self.df[col] = self.df[col].astype(np.int32)
                    elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                        self.df[col] = self.df[col].astype(np.int64)

                # Downcast Float types
                else:
                    # FIX: Cast the finfo limits to native Python floats to avoid float16 overflow warnings
                    f16_min = float(np.finfo(np.float16).min)
                    f16_max = float(np.finfo(np.float16).max)
                    f32_min = float(np.finfo(np.float32).min)
                    f32_max = float(np.finfo(np.float32).max)

                    if c_min > f16_min and c_max < f16_max:
                        self.df[col] = self.df[col].astype(np.float16)
                    elif c_min > f32_min and c_max < f32_max:
                        self.df[col] = self.df[col].astype(np.float32)
                    else:
                        self.df[col] = self.df[col].astype(np.float64)

            # Process Object/String columns
            else:
                if self.df[col].nunique() < 100:
                    self.df[col] = self.df[col].astype('category')

        end_mem = self.get_memory_usage()

        if verbose:
            decrease = 100 * (start_mem - end_mem) / start_mem
            print(f"--- Memory Optimization Result ---")
            print(f"Memory usage BEFORE: {start_mem:.2f} MB")
            print(f"Memory usage AFTER : {end_mem:.2f} MB")
            print(f"Decreased by       : {decrease:.2f}%")
            print(f"----------------------------------")

        return self.df


class EncoderProcessor(BaseDataProcessor):
    """
    Child class for handling categorical encoding (e.g., One-Hot Encoding).
    """
    def __init__(self, df: pd.DataFrame):
        super().__init__(df)

    def process(self):
        """
        Applies One-Hot Encoding to all categorical columns.
        Ensures 1/0 outputs instead of True/False and cleans up unused category levels.
        """
        # 1. Clean up unused categories in existing category columns
        # This prevents ghost columns from appearing (e.g., CODE_GENDER_XNA)
        for col in self.df.columns:
            if isinstance(self.df[col].dtype, pd.CategoricalDtype):
                self.df[col] = self.df[col].cat.remove_unused_categories()

        # 2. Identify all categorical columns ('object' or 'category' types)
        categorical_cols = [
            col for col in self.df.columns
            if (self.df[col].dtype == 'object' or isinstance(self.df[col].dtype, pd.CategoricalDtype))
        ]

        # 3. Apply One-Hot Encoding
        # Explicitly passing dtype=int forces pandas to output 1/0 instead of True/False
        self.df = pd.get_dummies(
            self.df,
            columns=categorical_cols,
            drop_first=True,
            dtype=int
        )

        return self.df

class ImputerProcessor(BaseDataProcessor):
    """
    Child class for handling missing value imputation for both numeric and categorical columns.
    """
    def __init__(self, df: pd.DataFrame, numeric_strategy: str = "median", categorical_strategy: str = "mode"):
        """
        Args:
            df (pd.DataFrame): The input pandas DataFrame.
            numeric_strategy (str): Strategy for numeric columns ('median' or 'mean'). Default is 'median'.
            categorical_strategy (str): Strategy for categorical columns ('mode' or 'constant'). Default is 'mode'.
        """
        super().__init__(df)
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy

    def process(self, verbose=True):
        """
        Iterates through columns and fills missing values based on the chosen strategies.
        """
        if verbose:
            start_missing = self.df.isnull().sum().sum()
            print(f"--- Missing Value Imputation ---")
            print(f"Total missing values BEFORE: {start_missing}")

        for col in self.df.columns:
            # Skip if the column has no missing values
            if not self.df[col].isnull().any():
                continue

            col_type = self.df[col].dtype

            # 1. Handle Numeric Columns (Integers and Floats)
            if col_type != object and not isinstance(col_type, pd.CategoricalDtype):
                if self.numeric_strategy == "median":
                    fill_value = self.df[col].median()
                elif self.numeric_strategy == "mean":
                    fill_value = self.df[col].mean()
                else:
                    fill_value = 0

                self.df[col] = self.df[col].fillna(fill_value)

            # 2. Handle Categorical Columns (Object or Category type)
            else:
                if self.categorical_strategy == "mode":
                    mode_series = self.df[col].mode()
                    # Fallback to 'Missing' if the entire column is NaN and mode is empty
                    fill_value = mode_series[0] if not mode_series.empty else "Missing"
                else:
                    fill_value = "Missing"

                # Safety check for pandas 'category' dtype:
                # If the fill_value is a new string, it must be added to categories metadata first
                if isinstance(col_type, pd.CategoricalDtype) and (fill_value not in self.df[col].cat.categories):
                    self.df[col] = self.df[col].cat.add_categories(fill_value)

                self.df[col] = self.df[col].fillna(fill_value)

        if verbose:
            end_missing = self.df.isnull().sum().sum()
            print(f"Total missing values AFTER : {end_missing}")
            print(f"--------------------------------")

        return self.df

def convert_days_to_years(df: pd.DataFrame, column_trans: list) -> pd.DataFrame:
    """
    Transforms specified columns from days to years, replaces the 'DAYS_' prefix
    with 'YEARS_', and drops the old columns.

    Args:
        df (pd.DataFrame): The input pandas DataFrame.
        column_trans (list): List of column names to transform (e.g., ['DAYS_BIRTH']).

    Returns:
        pd.DataFrame: The updated DataFrame with new 'YEARS_' columns.
    """
    # Create a copy to prevent changing the original dataframe out of scope
    df = df.copy()

    for col in column_trans:
        if col in df.columns:
            # Dynamically replace only the first occurrence of 'DAYS_' with 'YEARS_'
            # This perfectly preserves single or multiple suffixes (e.g., DAYS_A_B -> YEARS_A_B)
            new_col = col.replace("DAYS_", "YEARS_", 1)

            # Convert negative days to positive years using 365 days a year
            # Cast to float32 to keep memory usage low as optimized previously
            df[new_col] = (df[col].abs() / 365.0).astype('float32')

            # Drop the original day-based column to save space
            df.drop(columns=[col], inplace=True)
            print(f"Successfully transformed: '{col}' -> '{new_col}'")
        else:
            print(f"Warning: Column '{col}' was not found in the DataFrame.")

    return df


## App_train processing

In [ ]:
app_copy = app_train.copy()

# Drop value 'XNA' in CODE_GENDER
# Clean up the categorical metadata
app_copy = app_copy[app_copy['CODE_GENDER'] != "XNA"]
# app_copy['CODE_GENDER'] = app_copy['CODE_GENDER'].cat.remove_unused_categories()
print(f"App customer after drop 'XNA' value: {app_copy.shape}")

# Drop outlier in AMT_INCOME_TOTAL
app_copy = app_copy[app_copy['AMT_INCOME_TOTAL'] <= 1e8]
print(f"App customer after drop outlier income total: {app_copy.shape}")

#Initialize and inspect the original dataframe using the base class
base_processor = BaseDataProcessor(app_copy)
print(f"Original Shape: {base_processor.get_shape()}")
print(f"Original Memory: {base_processor.get_memory_usage():.2f} MB\n")





#